# OmniVoice Single Voice — Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doanquangkien/OmniVoice/blob/master/single-voice/colab_single_voice.ipynb)

> **GPU Miễn Phí:** T4 16GB · **Thời gian:** ~30s/audio · **Voice:** Nam công nghệ

## 📋 Hướng dẫn

1. **Runtime** → **Change runtime type** → **T4 GPU**
2. **Runtime** → **Run all** (hoặc Ctrl+F9)
3. Đợi ~3 phút → click link Gradio public
4. Dùng giao diện web để tạo giọng nói

> ⚠️ Phiên Colab tự ngắt sau ~90 phút không tương tác. GPU free ~4-6h/ngày.

## 1. Cài đặt + Tải voice_data.py

In [ ]:
# Cài dependencies (2-3 phút)
!pip install -q omnivoice gradio numpy torch torchaudio transformers accelerate soundfile librosa pydub
print('OK')

In [ ]:
# Tải voice_data.py từ GitHub (không cần upload thủ công)
!wget -q https://raw.githubusercontent.com/doanquangkien/OmniVoice/master/single-voice/voice_data.py
print('✅ voice_data.py downloaded!')

## 2. Khởi động

In [ ]:
import base64, logging, os, re, tempfile
import numpy as np
import torch
import gradio as gr
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device
from voice_data import VOICE_AUDIO_B64

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Load model
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True)
SAMPLING_RATE = model.sampling_rate
logger.info(f'Ready — SR: {SAMPLING_RATE}Hz')

VOICE_PROMPT = None
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=2.0,
    denoise=True, preprocess_prompt=True,
    postprocess_output=False,
    position_temperature=3.0,
    pad_duration=0.3, fade_duration=0.1,
)

def gen(text):
    global VOICE_PROMPT
    if VOICE_PROMPT is None:
        logger.info('Creating VoiceClonePrompt...')
        b = base64.b64decode(VOICE_AUDIO_B64)
        t = tempfile.NamedTemporaryFile(suffix='.mp3', delete=False)
        t.write(b); t.close()
        VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio=t.name)
        os.unlink(t.name)
        logger.info('VoiceClonePrompt cached!')
    
    text = text.strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs: return None
    if len(paragraphs) == 1:
        audio = model.generate(text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT, language='vi', speed=0.95, generation_config=GEN_CFG)[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(text=p, voice_clone_prompt=VOICE_PROMPT, language='vi', speed=0.95, generation_config=GEN_CFG)[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                audios.append(np.zeros(int(SAMPLING_RATE * 0.1)))
        audio = np.concatenate(audios)
    waveform = (audio * 32767).astype(np.int16)
    return (SAMPLING_RATE, waveform)

CSS = '.gradio-container{max-width:720px!important;margin:0 auto!important;padding:16px!important}footer{display:none!important}'
THEME = gr.themes.Soft(primary_hue='indigo')

with gr.Blocks(title='OmniVoice — Tạo giọng nói', theme=THEME, css=CSS) as demo:
    gr.Markdown('# OmniVoice\nChuyển văn bản thành giọng nói · Nam công nghệ')
    t = gr.Textbox(label='Nhập văn bản', lines=4, placeholder='Nhập văn bản bạn muốn chuyển thành giọng nói...')
    btn = gr.Button('Tạo giọng nói', variant='primary')
    out = gr.Audio(label='Kết quả')
    btn.click(gen, inputs=[t], outputs=[out], concurrency_limit=1)

demo.launch(server_name='0.0.0.0', server_port=7860, share=True, theme=THEME, css=CSS)